# Trabalhando com Arrays - Certificação DEA

## Dados do Exemplo
| customer_id | name | phones | addresses | purchase_history | preferences |
|-------------|------|--------|-----------|------------------|-------------|
| 424 | Larissa Nascimento | ["+55-48-92926-3606", "+55-62-99012-7693", "+55-56-93567-7851"] | [{"city":"Manaus","state":"AM","type":"home","zip":"69010-970"}, {"city":"Rio de Janeiro","state":"RJ","type":"work","zip":"22041-199"}] | [4299.5, 2489.27, 2335.6] | {"categories":["home","food","books"], "newsletter":false, "sms":false} |

---
```

## 1. EXPLODE vs EXPLODE_OUTER

### EXPLODE (retorna apenas registros com dados)
```sql
-- Explodir array de telefones
SELECT 
    customer_id,
    name,
    explode(phones) as phone
FROM techdados.bronze.customers
WHERE customer_id = 424;

-- Resultado: 3 linhas (1 por telefone)
-- customer_id | name               | phone
-- 424         | Larissa Nascimento | +55-48-92926-3606
-- 424         | Larissa Nascimento | +55-62-99012-7693
-- 424         | Larissa Nascimento | +55-56-93567-7851
```

### EXPLODE_OUTER (mantém registros mesmo se array é NULL/vazio)
```sql
SELECT 
    customer_id,
    name,
    explode_outer(phones) as phone
FROM techdados.bronze.customers;

-- Se phones for NULL ou [], ainda retorna 1 linha com phone = NULL
```

**📝 Certificação:** Questões perguntam a diferença entre EXPLODE e EXPLODE_OUTER!

## 2. Acessar Elementos de STRUCT (Notação de Ponto)
```sql
-- Acessar campos do struct preferences
SELECT 
    customer_id,
    preferences.newsletter as aceita_newsletter,
    preferences.sms as aceita_sms,
    preferences.categories as categorias_interesse
FROM techdados.bronze.customers
WHERE customer_id = 424;

-- Resultado:
-- customer_id | aceita_newsletter | aceita_sms | categorias_interesse
-- 424         | false             | false      | ["home","food","books",...]
```

## 3. POSEXPLODE - Explode com Posição ⭐
```sql
-- Explodir purchase_history com índice
SELECT 
    customer_id,
    name,
    pos as compra_numero,
    col as valor_compra
FROM techdados.bronze.customers
LATERAL VIEW posexplode(purchase_history) AS pos, col
WHERE customer_id = 424;

-- Resultado:
-- customer_id | name               | compra_numero | valor_compra
-- 424         | Larissa Nascimento | 0             | 4299.5
-- 424         | Larissa Nascimento | 1             | 2489.27
-- 424         | Larissa Nascimento | 2             | 2335.6
```

**PySpark:**
```python
from pyspark.sql.functions import posexplode

df = spark.table("techdados.bronze.customers")
df.select("customer_id", "name", posexplode("purchase_history")).show()
```

## 4. LATERAL VIEW - Flatten de Arrays ⭐⭐
```sql
-- Explodir addresses (array de struct)
SELECT 
    customer_id,
    name,
    addr.type as address_type,
    addr.city,
    addr.state,
    addr.zip
FROM techdados.bronze.customers
LATERAL VIEW explode(addresses) AS addr
WHERE customer_id = 424;

-- Resultado: 4 linhas (1 por endereço)
-- customer_id | name               | address_type | city          | state | zip
-- 424         | Larissa Nascimento | home         | Manaus        | AM    | 69010-970
-- 424         | Larissa Nascimento | work         | Rio de Janeiro| RJ    | 22041-199
-- 424         | Larissa Nascimento | delivery     | Fortaleza     | CE    | 60010-323
-- 424         | Larissa Nascimento | other        | Porto Alegre  | RS    | 90010-213
```

**PySpark:**
```python
from pyspark.sql.functions import explode, col

df = (spark.table("techdados.bronze.customers")
    .select("customer_id", "name", explode("addresses").alias("addr"))
    .select(
        "customer_id",
        "name", 
        col("addr.type").alias("address_type"),
        col("addr.city"),
        col("addr.state"),
        col("addr.zip")
    ))
```

## 5. INLINE - Flatten Arrays de Structs ⭐
```sql
-- Alternativa ao LATERAL VIEW para arrays de struct
SELECT 
    customer_id,
    name,
    inline(addresses)
FROM techdados.bronze.customers
WHERE customer_id = 424;

-- Resultado: expande direto as colunas do struct
-- customer_id | name               | city          | state | type     | zip
-- 424         | Larissa Nascimento | Manaus        | AM    | home     | 69010-970
-- 424         | Larissa Nascimento | Rio de Janeiro| RJ    | work     | 22041-199
```

**📝 Certificação:** INLINE é mais conciso que LATERAL VIEW para structs!

## 6. Múltiplos EXPLODE com LATERAL VIEW
```sql
-- Explodir phones E addresses (produto cartesiano!)
SELECT 
    customer_id,
    phone,
    addr.city
FROM techdados.bronze.customers
LATERAL VIEW explode(phones) AS phone
LATERAL VIEW explode(addresses) AS addr
WHERE customer_id = 424;

-- Resultado: 3 phones × 4 addresses = 12 linhas!
-- ⚠️ CUIDADO: Produto cartesiano pode gerar muitos registros
```

## 7. TRANSFORM - Manipular Arrays ⭐
```sql
-- Aplicar transformação em cada elemento do array
SELECT 
    customer_id,
    transform(purchase_history, x -> round(x, 0)) as valores_arredondados
FROM techdados.bronze.customers
WHERE customer_id = 424;

-- Resultado:
-- customer_id | valores_arredondados
-- 424         | [4300.0, 2489.0, 2336.0, 4136.0, 2264.0, 4622.0]
```

**PySpark:**
```python
from pyspark.sql.functions import transform, round as spark_round

df = spark.table("techdados.bronze.customers")
df = df.withColumn(
    "valores_arredondados",
    transform("purchase_history", lambda x: spark_round(x, 0))
)
```

## 8. FILTER em Arrays
```sql
-- Filtrar elementos do array
SELECT 
    customer_id,
    filter(purchase_history, x -> x > 3000) as compras_acima_3000
FROM techdados.bronze.customers
WHERE customer_id = 424;

-- Resultado:
-- customer_id | compras_acima_3000
-- 424         | [4299.5, 4135.89, 4621.83]
```

## Documentação Oficial

- [Funções de Array - Azure Databricks](https://learn.microsoft.com/pt-br/azure/databricks/sql/language-manual/sql-ref-functions-builtin#array-functions)
- [Higher-Order Functions](https://learn.microsoft.com/pt-br/azure/databricks/sql/language-manual/sql-ref-functions-builtin#higher-order-functions)
- [Lateral View - Azure Databricks](https://learn.microsoft.com/pt-br/azure/databricks/sql/language-manual/sql-ref-syntax-qry-select-lateral-view)